In [4]:
import cv2
from ultralytics import YOLO
from collections import deque
import numpy as np

# ✅ 用你新训练的模型
model = YOLO(r"C:\Users\foowe\Documents\Yolo\final_models\emotion_detect_final_best.pt")

# 摄像头
cap = cv2.VideoCapture(0)

# 用来做时间平滑（关键）
emotion_history = deque(maxlen=20)

# 定义“抑郁相关情绪”
depression_emotions = ["sad", "fear", "anger"]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.25, verbose=False)

    current_emotions = []

    for r in results:
        boxes = r.boxes
        if boxes is None:
            continue

        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id]

            current_emotions.append(label)

            # 画框
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            text = f"{label} {conf:.2f}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, text, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # 存入历史
    if current_emotions:
        emotion_history.extend(current_emotions)

    # === 计算抑郁倾向 ===
    if len(emotion_history) > 0:
        total = len(emotion_history)
        dep_count = sum(1 for e in emotion_history if e in depression_emotions)

        depression_score = dep_count / total

        status = "Depressed" if depression_score > 0.6 else "Normal"

        cv2.putText(frame, f"Depression Score: {depression_score:.2f}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

        cv2.putText(frame, f"Status: {status}",
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()